In [ ]:
# ── Cell 1: Imports & device ──────────────────────────────────────────────────
import copy, json, math, os, time, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Enable TF32 for speed on Ampere+ GPUs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('⚠️  No GPU — running on CPU. Go to Runtime → Change runtime type → T4 GPU')

Device: cuda
GPU: Tesla T4


In [ ]:
# ── Cell 2: Config — edit here ────────────────────────────────────────────────
TOTAL_STEPS  = 5000     # gradient steps per run (reduce to 500 for a quick test)
N_SEEDS      = 5        # seeds (reduce to 1 or 2 for faster testing)
BATCH_SIZE   = 64
BLOCK_SIZE   = 256      # context length in characters
EVAL_ITERS   = 50       # val batches averaged per evaluation
LOG_EVERY    = 100      # log every N steps
DATA_DIR     = './data'

# Model size — reduce for faster runs
N_LAYER  = 6
N_HEAD   = 6
N_EMBD   = 384
DROPOUT  = 0.2

ADAM_LR_GRID = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]

# ── Ultra v5 hyperparams (DO NOT CHANGE — identical to raw-fn benchmark) ──────
# ── Ultra v5 hyperparams — GPT-tuned ──────────────────────────────────────────
LR_MAX           = 3e-4    # transformer sweet spot; 5e-3 is too aggressive
LR_MIN           = 1e-5
T_CYCLE          = 99999   # effectively disable cycling — no more LR resets
EXPLORE_FRAC     = 0.0     # skip the cyclic explore phase entirely; go straight to cosine
LR_EXPLOIT_START = 3e-4    # cosine decay from here...
LR_EXPLOIT_END   = 3e-5    # ...to here (10x decay over full run)
EXPLORE_DECAY_START = 0.0  # ef=1.0 never used since explore phase is off
EXPLORE_FLOOR       = 1.0
TANG_BETA        = 0.80
TANG_EPS_BASE    = 4e-4
TANG_INTERVAL    = 100
TANG_LOSS_GATE   = 0
SHARP_UPDATE_INT = 200
FD_EPS           = 1e-4
NOISE_SCALE      = 1e-5    # 20x smaller — GPT is stochastic already; noise hurts
NOISE_START      = 99999   # effectively disable noise injection
print('Config loaded.')

Config loaded.


In [ ]:
# ── Cell 3: Download & prepare Tiny Shakespeare ───────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print(f'Downloading Tiny Shakespeare...')
    urllib.request.urlretrieve(URL, DATA_PATH)
    print('  done.')

with open(DATA_PATH, 'r') as f:
    text = f.read()

chars  = sorted(set(text))
VOCAB  = len(chars)
stoi   = {c: i for i, c in enumerate(chars)}
itos   = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

data       = torch.tensor(encode(text), dtype=torch.long)
n_train    = int(0.9 * len(data))
train_data = data[:n_train]
val_data   = data[n_train:]

print(f'Dataset: {len(text):,} chars | vocab={VOCAB} | '
      f'train={len(train_data):,} | val={len(val_data):,}')

def get_batch(split, rng):
    source = train_data if split == 'train' else val_data
    ix = rng.integers(0, len(source) - BLOCK_SIZE, size=(BATCH_SIZE,))
    x  = torch.stack([source[i   : i+BLOCK_SIZE  ] for i in ix]).to(DEVICE)
    y  = torch.stack([source[i+1 : i+BLOCK_SIZE+1] for i in ix]).to(DEVICE)
    return x, y

  done.
Dataset: 1,115,394 chars | vocab=65 | train=1,003,854 | val=111,540


In [ ]:
# ── Cell 4: NanoGPT model (LayerNorm only — no BatchNorm) ────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head  = n_head
        self.n_embd  = n_embd
        self.c_attn  = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj  = nn.Linear(n_embd, n_embd)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer('bias',
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        return self.resid_drop(self.c_proj((att @ v).transpose(1, 2).contiguous().view(B, T, C)))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd), nn.GELU(),
            nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.mlp  = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, N_EMBD)
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop    = nn.Dropout(DROPOUT)
        self.blocks  = nn.Sequential(*[Block(N_EMBD, N_HEAD, BLOCK_SIZE, DROPOUT)
                                        for _ in range(N_LAYER)])
        self.ln_f    = nn.LayerNorm(N_EMBD)
        self.head    = nn.Linear(N_EMBD, VOCAB, bias=False)
        self.tok_emb.weight = self.head.weight   # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.LayerNorm): nn.init.zeros_(m.bias); nn.init.ones_(m.weight)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos  = torch.arange(T, device=idx.device).unsqueeze(0)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x    = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss   = F.cross_entropy(logits.view(-1, VOCAB), targets.view(-1)) if targets is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -BLOCK_SIZE:])
            next_tok  = torch.multinomial(F.softmax(logits[:, -1, :] / temperature, dim=-1), 1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

n_params = sum(p.numel() for p in NanoGPT().parameters())
print(f'NanoGPT ready — {n_params/1e6:.2f}M parameters')

NanoGPT ready — 10.77M parameters


In [ ]:
# ── Cell 5: Ultra v5 core helpers ─────────────────────────────────────────────
def cyclic_lr(step, T, lr_max, lr_min):
    return lr_min + 0.5*(lr_max-lr_min)*(1+np.cos(np.pi*(step%T)/T))

def cosine_decay_lr(step, total, lr_start, lr_end):
    p = min(step/max(total,1), 1.0)
    return lr_end + 0.5*(lr_start-lr_end)*(1+np.cos(np.pi*p))

def get_ef(step, total, decay_start_frac, floor):
    ds = int(total*decay_start_frac)
    if step < ds: return 1.0
    return max(floor, 1.0-(step-ds)/max(total-ds,1))

def get_flat_grad(model):
    return np.concatenate([
        p.grad.detach().cpu().float().view(-1).numpy() if p.grad is not None
        else np.zeros(p.numel(), np.float32)
        for p in model.parameters()
    ]).astype(np.float32)

def get_flat_params(model):
    return np.concatenate(
        [p.detach().cpu().float().view(-1).numpy() for p in model.parameters()]
    ).astype(np.float32)

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(torch.tensor(flat[offset:offset+n], dtype=p.dtype,
                                   device=p.device).view(p.shape))
        offset += n

def fd_curvature_nn(model, xb, yb, g_np, eps=FD_EPS):
    norm_g = np.linalg.norm(g_np)
    if norm_g < 1e-12: return 0.0
    norm_g = min(norm_g, 1.0)
    g_hat   = g_np / norm_g
    params0 = get_flat_params(model)
    with torch.no_grad(): _, l0 = model(xb, yb); f0 = l0.item()
    set_flat_params(model, params0 + g_hat*eps)
    with torch.no_grad(): _, l1 = model(xb, yb); f1 = l1.item()
    set_flat_params(model, params0)
    return float((f1 - f0 - eps*norm_g) / (0.5*eps**2 + 1e-30))

class TangentMomentum:
    def __init__(self, dim, beta=TANG_BETA):
        self.beta = beta; self.v = np.zeros(dim, np.float32)
    def step(self, model, g_np, epsilon):
        g_hat = g_np / (np.linalg.norm(g_np)+1e-8)
        noise = np.random.randn(len(g_np)).astype(np.float32)
        noise -= np.dot(noise, g_hat)*g_hat
        noise /= np.linalg.norm(noise)+1e-8
        self.v = self.beta*self.v + (1-self.beta)*noise
        direction = self.v / (np.linalg.norm(self.v)+1e-8)
        set_flat_params(model, get_flat_params(model) + direction*epsilon)

@torch.no_grad()
def estimate_val_loss(model, rng):
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch('val', rng)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

print('Ultra v5 core ready.')

Ultra v5 core ready.


In [ ]:
# ── Cell 6: run_ultra & run_adam ──────────────────────────────────────────────
def run_ultra(seed, total_steps):
    torch.manual_seed(seed); np.random.seed(seed)
    rng   = np.random.default_rng(seed)
    model = NanoGPT().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR_MAX, betas=(0.9,0.999), weight_decay=0.0)

    n_params    = sum(p.numel() for p in model.parameters())
    tang_mom    = TangentMomentum(dim=n_params)
    explore_end = int(total_steps * EXPLORE_FRAC)

    best_val_loss = float('inf')
    best_state    = copy.deepcopy(model.state_dict())
    sharpness = tang_exec = tang_block = 0
    history = []

    print(f'  [Ultra seed={seed}] {n_params/1e6:.2f}M params | {total_steps} steps')
    for step in range(total_steps):
        # LR schedule
        if step < explore_end:
            lr = cyclic_lr(step, T_CYCLE, LR_MAX, LR_MIN)
        else:
            lr = cosine_decay_lr(step-explore_end, total_steps-explore_end,
                                  LR_EXPLOIT_START, LR_EXPLOIT_END)
        for pg in opt.param_groups: pg['lr'] = lr
        ef = get_ef(step, total_steps, EXPLORE_DECAY_START, EXPLORE_FLOOR)

        # gradient step
        xb, yb = get_batch('train', rng)
        opt.zero_grad()
        _, loss = model(xb, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        g_np = get_flat_grad(model)
        opt.step()
        loss_val = loss.item()

        # noise
        if step >= NOISE_START:
            sigma = NOISE_SCALE * np.sqrt(max(loss_val, 1e-8)) * ef
            with torch.no_grad():
                for p in model.parameters(): p.data.add_(torch.randn_like(p)*sigma)

        # sharpness
        if step % SHARP_UPDATE_INT == 0 and step > 0:
            sharpness = fd_curvature_nn(model, xb, yb, g_np)

        # tangent step
        if step % TANG_INTERVAL == 0 and step > 500:
            loss_gate  = loss_val > TANG_LOSS_GATE * max(best_val_loss, 1e-10)
            sharp_gate = sharpness < 0.8*(2.0/max(lr,1e-8))
            if loss_gate and sharp_gate:
                sharp_safe = max(abs(sharpness), 1.0)
                eps_tang   = float(np.clip(TANG_EPS_BASE*(50.0/sharp_safe)**0.5, 1e-5, 5e-3))*ef
                tang_mom.step(model, g_np, eps_tang)
                tang_exec += 1
            else:
                tang_block += 1

        # logging
        if step % LOG_EVERY == 0:
            val_loss = estimate_val_loss(model, rng)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = copy.deepcopy(model.state_dict())
            history.append({'step': step, 'train_loss': float(loss_val),
                            'val_loss': float(val_loss), 'lr': float(lr), 'ef': float(ef)})
            print(f'    step {step:5d}/{total_steps}  train={loss_val:.4f}  '
                  f'val={val_loss:.4f}  lr={lr:.2e}  ef={ef:.2f}  sharp={sharpness:.2f}')

    model.load_state_dict(best_state)
    final_val = estimate_val_loss(model, rng)
    ppl       = math.exp(min(final_val, 20))
    context   = torch.zeros((1,1), dtype=torch.long, device=DEVICE)
    sample    = decode(model.generate(context, 300, temperature=0.8)[0].tolist())
    return {'optimizer': 'ultra_v5', 'seed': seed, 'final_val_loss': float(final_val),
            'best_val_loss': float(best_val_loss), 'perplexity': float(ppl),
            'tang_exec': tang_exec, 'tang_block': tang_block,
            'history': history, 'sample': sample}


def run_adam_single(seed, lr, total_steps):
    torch.manual_seed(seed); np.random.seed(seed)
    rng   = np.random.default_rng(seed)
    model = NanoGPT().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.999), weight_decay=0.0)
    history = []
    for step in range(total_steps):
        xb, yb = get_batch('train', rng)
        opt.zero_grad()
        _, loss = model(xb, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % LOG_EVERY == 0:
            val_loss = estimate_val_loss(model, rng)
            history.append({'step': step, 'train_loss': float(loss.item()),
                            'val_loss': float(val_loss)})
    final_val = estimate_val_loss(model, rng)
    ppl       = math.exp(min(final_val, 20))
    context   = torch.zeros((1,1), dtype=torch.long, device=DEVICE)
    sample    = decode(model.generate(context, 300, temperature=0.8)[0].tolist())
    return {'optimizer': f'adam_lr{lr:.0e}', 'lr': lr, 'seed': seed,
            'final_val_loss': float(final_val), 'perplexity': float(ppl),
            'history': history, 'sample': sample}

def run_adam_best(seed, total_steps):
    best = None
    for lr in ADAM_LR_GRID:
        r = run_adam_single(seed, lr, total_steps)
        print(f'  [Adam seed={seed} lr={lr:.0e}]  val_loss={r["final_val_loss"]:.4f}  ppl={r["perplexity"]:.2f}')
        if best is None or r['final_val_loss'] < best['final_val_loss']: best = r
    return best

print('Runner functions ready.')

Runner functions ready.


In [ ]:
# ── Cell 7: RUN ───────────────────────────────────────────────────────────────
print('=' * 68)
print('  ULTRA BENCHMARK SUITE — SHAKESPEARE GPT EDITION')
print(f'  {TOTAL_STEPS} steps | {N_SEEDS} seeds | {len(ADAM_LR_GRID)} Adam LRs | device={DEVICE}')
print(f'  Model: {N_LAYER}L/{N_HEAD}H/{N_EMBD}E | block={BLOCK_SIZE} | vocab={VOCAB}')
print('=' * 68)

ultra_all, adam_all = [], []
u_losses, a_losses, u_ppls, a_ppls = [], [], [], []
grand_start = time.time()

for seed in range(N_SEEDS):
    print(f'\n{"─"*68}\n  SEED {seed}\n{"─"*68}')
    t0 = time.time()

    ur = run_ultra(seed, TOTAL_STEPS)
    ultra_all.append(ur); u_losses.append(ur['final_val_loss']); u_ppls.append(ur['perplexity'])

    ar = run_adam_best(seed, TOTAL_STEPS)
    adam_all.append(ar); a_losses.append(ar['final_val_loss']); a_ppls.append(ar['perplexity'])

    winner = 'U' if ur['final_val_loss'] < ar['final_val_loss'] else \
             ('A' if ar['final_val_loss'] < ur['final_val_loss'] else '=')
    print(f'\n  seed={seed}  Ultra val={ur["final_val_loss"]:.4f} ppl={ur["perplexity"]:.2f}  ||  '
          f'Adam val={ar["final_val_loss"]:.4f} ppl={ar["perplexity"]:.2f}  [{winner}]  {time.time()-t0:.1f}s')
    print(f'\n  ── Ultra sample ──\n{ur["sample"][:250]}')
    print(f'\n  ── Adam  sample ──\n{ar["sample"][:250]}')

print(f'\nTotal wall time: {time.time()-grand_start:.1f}s')

  ULTRA BENCHMARK SUITE — SHAKESPEARE GPT EDITION
  5000 steps | 5 seeds | 5 Adam LRs | device=cuda
  Model: 6L/6H/384E | block=256 | vocab=65

────────────────────────────────────────────────────────────────────
  SEED 0
────────────────────────────────────────────────────────────────────
  [Ultra seed=0] 10.77M params | 5000 steps
    step     0/5000  train=4.2705  val=3.6769  lr=3.00e-04  ef=1.00  sharp=0.00
    step   100/5000  train=2.5113  val=2.4721  lr=3.00e-04  ef=1.00  sharp=0.00
    step   200/5000  train=2.4253  val=2.4146  lr=2.99e-04  ef=1.00  sharp=264479.12
    step   300/5000  train=2.3048  val=2.2788  lr=2.98e-04  ef=1.00  sharp=264479.12
    step   400/5000  train=2.1380  val=2.1228  lr=2.96e-04  ef=1.00  sharp=-293592.88
    step   500/5000  train=1.9932  val=2.0040  lr=2.93e-04  ef=1.00  sharp=-293592.88
    step   600/5000  train=1.8993  val=1.9156  lr=2.91e-04  ef=1.00  sharp=658086.06
    step   700/5000  train=1.8077  val=1.8503  lr=2.87e-04  ef=1.00  sharp=658